# Step 1 — Feature extraction and matching, on a Colab GPU

This notebook does the **GPU-heavy half** of photogrammetry: it finds features in
your photos, then matches them across image pairs. The output is a single
`database.db` file.

After this, run **step 2** (the desktop app) to build camera positions, then
**step 3** to train Gaussian Splatting.

**Before you run anything:** `Runtime` → `Change runtime type` → **T4 GPU**.

### Why this notebook exists

The `colmap` package from `apt` is compiled **without CUDA**. On CPU, exhaustive
matching of 320 photos takes over 11 hours. On a T4 it takes 28 minutes — the
same work, about 23× faster per image pair.

This notebook uses a prebuilt COLMAP 3.13.0 with CUDA enabled:
https://github.com/Ryanhuhut/colmap-cuda-colab

## 1. Configuration — edit this cell

In [ ]:
# ==================== EDIT THIS CELL ====================

# Zip file on Google Drive holding your photos.
# The photos must sit directly inside the zip, not in a sub-folder.
IMAGES_ZIP = "/content/drive/MyDrive/meo_no_input.zip"

# Folder on Drive where the finished database will be saved.
OUTPUT_DIR = "/content/drive/MyDrive/img3dpl"

# "exhaustive"  compares every possible pair of photos. Best quality — it catches
#               loop closures, so the model does not drift. Needs a GPU.
# "sequential"  compares each photo with its 10 neighbours only. Roughly 17x less
#               work, but it misses loop closures when you orbit an object.
MATCHER = "exhaustive"

# Camera model. OPENCV suits phone cameras.
CAMERA_MODEL = "OPENCV"

# True when every photo came from the same camera at the same zoom.
SINGLE_CAMERA = True

# ========================================================
print("Photos  :", IMAGES_ZIP)
print("Output  :", OUTPUT_DIR)
print("Matcher :", MATCHER)

## 2. Setup — GPU, COLMAP, photos

In [ ]:
import os
import subprocess
import sys

COLMAP_URL = ("https://github.com/Ryanhuhut/colmap-cuda-colab/releases/latest/"
              "download/colmap-3.13.0-cuda12.2-ubuntu2204-sm75.tar.gz")
IMAGES_DIR = "/content/images"

os.environ["PATH"] = "/opt/colmap-cuda/bin:" + os.environ["PATH"]

# ---- GPU -------------------------------------------------------------------
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version",
                      "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU attached. Runtime > Change runtime type > T4 GPU.")

gpu_name = gpu.stdout.strip()
print("GPU:", gpu_name)

# This COLMAP build targets sm_75 only, which is the T4. On any other card it
# fails with a CUDA architecture error. Better to say so now than 20 minutes in.
if "T4" not in gpu_name:
    print()
    print("!! WARNING — this COLMAP build only supports the Tesla T4 (sm_75).")
    print(f"!! You were given: {gpu_name}")
    print("!! Matching will fail. Rebuild COLMAP with CUDA_ARCH=\"75;80;89\";")
    print("!! see build-colmap/ in the repository.")

# ---- COLMAP ----------------------------------------------------------------
if not os.path.exists("/opt/colmap-cuda/bin/colmap"):
    print("\nInstalling COLMAP (about 7 seconds)...")
    subprocess.run(["wget", "-q", COLMAP_URL, "-O", "/tmp/colmap.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/colmap.tar.gz", "-C", "/opt"], check=True)

version = subprocess.run(["colmap", "-h"], capture_output=True, text=True).stdout
print("\n".join(version.splitlines()[:2]))
if "with CUDA" not in version:
    raise SystemExit("This COLMAP has no CUDA support — wrong package?")

# ---- Photos ----------------------------------------------------------------
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

if not os.path.isdir(IMAGES_DIR) or not os.listdir(IMAGES_DIR):
    if not os.path.isfile(IMAGES_ZIP):
        raise SystemExit(f"Zip not found on Drive: {IMAGES_ZIP}")
    print("\nExtracting photos...")
    os.makedirs(IMAGES_DIR, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", IMAGES_ZIP, "-d", IMAGES_DIR], check=True)

PHOTOS = sorted(f for f in os.listdir(IMAGES_DIR)
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff")))
N_PHOTOS = len(PHOTOS)
print(f"\nPhotos ready: {N_PHOTOS}")
if N_PHOTOS == 0:
    raise SystemExit("No photos found. Check IMAGES_ZIP, and that the zip is flat.")
if N_PHOTOS < 20:
    print("Only a handful of photos — reconstruction usually needs 20-30 at least.")

# Rough idea of the work ahead, so nobody thinks it has frozen.
pairs = N_PHOTOS * (N_PHOTOS - 1) // 2 if MATCHER == "exhaustive" else N_PHOTOS * 10
print(f"Pairs to match: {pairs:,}   (about {pairs * 0.033 / 60:.0f} minutes on a T4)")

## 3. Extract features and match

Output is streamed live, so you can watch it move. Do not close this tab.

`exhaustive` matching on 320 photos took **28 minutes** on a free-tier T4.

In [ ]:
import os
import shutil
import subprocess
import time

WORK = "/content/colmap_work"
DB = f"{WORK}/database.db"

# Start from an empty folder every time. Reusing a database built by a different
# COLMAP version, or by a different set of photos, produces confusing
# "SQL logic error" failures much later on.
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)


def run(label, cmd):
    """Run a COLMAP command and stream its output live."""
    print(f"\n{'=' * 64}\n{label}\n{'=' * 64}", flush=True)
    started = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    minutes = (time.time() - started) / 60
    print(f"\n>>> {label} — {minutes:.1f} min, exit code {code}", flush=True)
    if code != 0:
        raise RuntimeError(f"{label} failed. Read the output above.")
    return minutes


t_extract = run("1 of 2 — Feature extraction", [
    "colmap", "feature_extractor",
    "--database_path", DB,
    "--image_path", IMAGES_DIR,
    "--ImageReader.camera_model", CAMERA_MODEL,
    "--ImageReader.single_camera", "1" if SINGLE_CAMERA else "0",
    "--FeatureExtraction.use_gpu", "1",
])

t_match = run(f"2 of 2 — Matching ({MATCHER})", [
    "colmap", f"{MATCHER}_matcher",
    "--database_path", DB,
    "--FeatureMatching.use_gpu", "1",
])

print(f"\nTotal: {t_extract + t_match:.1f} minutes")

## 4. Save the database to Drive

This runs automatically. The database lives on the Colab machine, which is wiped
the moment the session ends — so it is copied to Drive straight away.

In [ ]:
import os
import shutil
import sqlite3

os.makedirs(OUTPUT_DIR, exist_ok=True)
saved = os.path.join(OUTPUT_DIR, "database.db")
shutil.copy(DB, saved)

with sqlite3.connect(f"file:{saved}?mode=ro", uri=True) as conn:
    images = conn.execute("SELECT COUNT(*) FROM images").fetchone()[0]
    pairs = conn.execute("SELECT COUNT(*) FROM two_view_geometries").fetchone()[0]

print(f"Saved: {saved}")
print(f"  size   : {os.path.getsize(saved) / 1e9:.1f} GB")
print(f"  images : {images}")
print(f"  pairs  : {pairs:,}")
print()
print("Next: download this file, then open the desktop app to build camera")
print("positions. Building positions runs on the CPU — a GPU does not help there,")
print("and your own machine most likely has more CPU cores than free-tier Colab.")